In [1]:
from py_module.metadata.connection.ConnectionRegistry import PostgresConnection as pgconn
from py_module.metadata.render_jinja.RenderRegistry import ChangeDataCaptureExtraction as cdc, RetrieveSchema as rs, SchemaSource as ss 
from py_module.metadata.connection.StorageBaseConnection import StorageConn as storage
from py_module.metadata.datatype_conversion.Avro import DataTypeConverter as dtc
from py_module.exec.SourceToAvro import ExecuteGCS
from sqlalchemy import text
import json
from fastavro import writer, parse_schema
from datetime import datetime as dt, timedelta


# SETUP (connection + get template)

In [2]:
params = {
    "database":"postgres",
    "db": "meteo",
    "host": "localhost",
    "port": 5433,
    "user": "meteo",
    "password": "meteo",
    "ssl_args": {}
} 

In [3]:
engine = pgconn.get_engine(**params)
conn = engine.connect()

In [4]:
cdc_extraction = cdc.render_jinja(database=params['database'])
retrieve_schema = rs.render_jinja(database=params['database'])
schema_source = ss.render_jinja()

# Render template with appropriate values

### Retrieve schema table -- this is useful for next steps

In [5]:
table_schema = 'public'
table_name = 'fct_meteo'
rendered_schema = retrieve_schema.render(
                                table_schema=table_schema,
                                table_name=table_name
                            )

In [6]:
it_schema = conn.execute(text(rendered_schema))
converter = dtc()  # your DataTypeConverter instance, optionally pass defaults

dbt_columns = list()
avro_columns = list()
cdc_columns = list()

for i in it_schema:
    col_name = i[0]
    db_type = i[1]
    precision = i[2]
    scale = i[3]

    avro_type = converter.source_to_avro(params['database'], db_type, numeric_precision=precision, numeric_scale=scale)
    bq_type = converter.source_to_bigquery(params['database'], db_type)

    # setup for cdc model injection
    cdc_columns.append(col_name)

    # setup the columns for avro injection with default
    avro_field = converter.generate_avro_field(col_name, avro_type)
    avro_columns.append(avro_field)

    # setup the columns for dbt source
    dbt_columns.append({col_name: bq_type})


In [7]:

rendered_sources = schema_source.render(
    schema_name = table_schema,
    table_name = table_name,
    cols = dbt_columns,
    database = params['database'],
    staging_dataset = 'staging',
    istance_name = params['db'],
    bucket_name = 'postgres__d-meteo-db',
    version = 'v1'
    )

### define custom CDC extraction query 

In [8]:
rendered_cdc = cdc_extraction.render(
                                columns = cdc_columns,
                                schema_name = table_schema, 
                                table_name = table_name, 
                                delta = False, 
                                # delta_column = delta_column, 
                                # delta_timestamp = delta_timestamp, 
                                # where_conditions = where_conditions
                            )

In [9]:
cdc_exec = conn.execution_options(stream_results=True).execute(text(rendered_cdc))

In [10]:
executor = ExecuteGCS(
    table_schema = table_schema,
    table_name = table_name,
    bucket_name = 'postgres__d-meteo-db'
)

In [11]:
executor.yield_chunks(
    avro_columns = avro_columns,
    cdc_executor = cdc_exec
)

### Push sources.yml file and avro schema